# **Fashion-MNIST Classification (MLP with TensorFlow/Keras)**

In [ ]:
# Libraries imported
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras import optimizers, losses
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

## Data Loading & Preprocessing

In [ ]:
# Load data directly from keras.datasets
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# Preprocessing (Normalize to [-1, 1] as in the original PyTorch transforms)
# Original was ToTensor() [0, 255] -> [0, 1] then Normalize((0.5,), (0.5,)) [0, 1] -> [-1, 1]
x_train = (x_train.astype('float32') / 255.0) * 2.0 - 1.0
x_test = (x_test.astype('float32') / 255.0) * 2.0 - 1.0

# Data loaders (not needed for model.fit, but batch_size is)
batch_size = 64

# Class labels
classes = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
           'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

## Define Neural Network Model

In [ ]:
model = Sequential([
    Flatten(input_shape=(28, 28)),  # Replaces x.view(-1, 28*28)
    Dense(256, activation='relu'),  # Replaces fc1 + relu
    Dense(128, activation='relu'),  # Replaces fc2 + relu
    Dense(10)                         # Replaces fc3. Output are logits.
])

# Initialize loss and optimizer
# Use SparseCategoricalCrossentropy with from_logits=True to match PyTorch's CrossEntropyLoss
criterion = losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = optimizers.Adam(learning_rate=0.001)

# Compile the model
model.compile(optimizer=optimizer,
              loss=criterion,
              metrics=['accuracy'])

model.summary()

## Training the Model

In [ ]:
epochs = 10

# Train the model using model.fit()
history = model.fit(x_train, y_train,
                    epochs=epochs,
                    batch_size=batch_size,
                    shuffle=True,
                    verbose=1)

# Extract data for plotting
train_losses = history.history['loss']
train_accuracies = [acc * 100 for acc in history.history['accuracy']]


## Final Evaluation

In [ ]:
# Use model.evaluate() for final accuracy
test_loss, final_accuracy = model.evaluate(x_test, y_test, verbose=0)

print(f"\n✅ Final Test Accuracy: {final_accuracy * 100:.2f}%")

# Get predictions for confusion matrix
predictions = model.predict(x_test)
y_pred = np.argmax(predictions, axis=1)
y_true = y_test


## Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

## Visualize Sample Predictions

In [ ]:
# Number of images to display
n_images = 14

# Get a batch of test data (just slice the numpy arrays)
images, labels = x_test[:n_images], y_test[:n_images]

# Get predictions
outputs = model.predict(images)
preds = np.argmax(outputs, axis=1)

# Create figure
plt.figure(figsize=(14, 6))
for i in range(n_images):
    plt.subplot(2, n_images // 2, i + 1)

    # Undo normalization: from [-1,1] → [0,1]
    img = images[i] * 0.5 + 0.5
    # .squeeze() is not needed, images are already (28, 28)

    # Plot grayscale image
    plt.imshow(img, cmap='gray')

    # Title shows prediction and true label
    color = 'green' if preds[i] == labels[i] else 'red'
    plt.title(f"Pred: {classes[preds[i]]}\nTrue: {classes[labels[i]]}",
              color=color, fontsize=10)
    plt.axis('off')

plt.suptitle("Model Predictions on Test Images", fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

## Plot Training Progress

In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(train_losses, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss over Epochs')
plt.legend()

plt.subplot(1,2,2)
plt.plot(train_accuracies, label='Train Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Train Accuracy over Epochs')
plt.legend()
plt.show()

## Conclusion

The fully connected neural network achieved a final test accuracy of 88.5%, surpassing the target benchmark of 85%. The training loss consistently decreased across epochs, indicating effective learning and stable convergence. From the confusion matrix, the model performed particularly well on classes like Sneaker, Bag, and Trouser, while showing some confusion between visually similar categories such as Shirt, T-shirt/top, and Coat. Overall, the model demonstrates strong classification performance on the Fashion-MNIST dataset, effectively distinguishing most clothing types using a simple multilayer perceptron architecture.